# Linked Lists + Backtracking + Greedy + Math + Bit Manipulation — Combined Masterclass

The ninth and final notebook in the DSA series. These five topics are **less central** than what's come before, but they pop up in interviews enough that going in unprepared is risky. The strategy here is depth-on-fundamentals plus breadth-on-canonical-problems — enough to handle whatever the interview throws at you.

**Why combine these five?**
- **Linked lists** — 8-10 canonical patterns, very interview-friendly. You either know them or you don't.
- **Backtracking** — one universal template; once you understand it, all 20+ problems collapse into variants.
- **Greedy** — algorithmically simple but conceptually subtle. The hard part is *recognizing* when greedy works.
- **Math** — GCD, primes, modular arithmetic, fast power. Small surface area, classic problems.
- **Bit manipulation** — six operators and a half-dozen tricks. Tiny topic with high "appears out of nowhere" risk.

Together they cover the long tail of "what if they ask something I haven't drilled?"

## How to read this notebook

This notebook is structured as **five self-contained chapters** plus a final synthesis. You can read them in any order, but the suggested order is:
1. **Linked Lists** (§1-3) — most likely to come up in a real interview round
2. **Backtracking** (§4-6) — high-frequency in Google-style interviews
3. **Greedy** (§7-9) — high signal because the *reasoning* is what's tested
4. **Math** (§10-12) — small topic, high-trap potential
5. **Bit manipulation** (§13-15) — smallest topic, most "Oh, that's a clever trick" wow factor
6. **Synthesis** (§16) — cross-topic pattern recognition, master complexity table, common bugs

## Topic-level cheat sheets

### Linked List — when to reach for which technique
| Signal | Technique |
|---|---|
| Edge cases involving head | Dummy node |
| Find midpoint, detect cycle, find kth-from-end | Slow / fast pointers |
| Reverse all or part of a list | Iterative or recursive reverse |
| Merge / interleave / split | Multi-pointer with dummy head |
| In-place check (palindrome) | Reverse second half |

### Backtracking — the universal template
```
def backtrack(state):
    if is_solution(state): record(state); return
    for choice in choices(state):
        if not valid(choice, state): continue
        apply(choice, state)
        backtrack(state)
        undo(choice, state)        # ← the "back" in backtracking
```

### Greedy — when does greedy WORK?
A greedy choice yields a globally optimal solution iff the problem has the **greedy choice property** (local optimum extends to global) AND **optimal substructure** (subproblems optimally combine). The interview skill is proving (or convincingly arguing) the exchange argument.

### Math — the 5 things you must know cold
1. GCD via Euclidean algorithm, and LCM = a*b/gcd
2. Fast modular exponentiation in O(log n)
3. Sieve of Eratosthenes for all primes up to n in O(n log log n)
4. Prime factorization in O(√n)
5. Modular arithmetic basics: `(a+b) % m`, `(a*b) % m`, modular inverse for division

### Bit manipulation — the 8 essential tricks
| Operation | Code |
|---|---|
| Check bit i | `(n >> i) & 1` |
| Set bit i | `n | (1 << i)` |
| Clear bit i | `n & ~(1 << i)` |
| Toggle bit i | `n ^ (1 << i)` |
| Clear lowest set bit | `n & (n - 1)` |
| Isolate lowest set bit | `n & -n` |
| Check power of 2 | `n > 0 and (n & (n - 1)) == 0` |
| XOR all = find odd-count | `for x in arr: result ^= x` |


# Part 1 — Linked Lists

## 1. Foundations

A **linked list** is a chain of nodes where each node holds data and a pointer to the next node. Compared to an array:

| Operation | Array | Singly linked list |
|---|---|---|
| Access by index | O(1) | O(n) |
| Search for value | O(n) | O(n) |
| Insert at front | O(n) (shift everything) | O(1) (rewire head) |
| Insert at back | O(1) amortized | O(n) (walk to tail) or O(1) with tail pointer |
| Insert at middle | O(n) (shift) | O(1) **given the predecessor pointer** |
| Delete at front | O(n) | O(1) |
| Delete at back | O(1) | O(n) |
| Memory layout | Contiguous (cache friendly) | Scattered (cache unfriendly) |

**Why linked lists in interviews?** They're the cleanest test of pointer manipulation. You can't fake your way through a "reverse this list" answer — either the prev/curr/next dance works or it doesn't. The patterns transfer directly to graph and tree problems.

### The 5 things that go wrong with linked list code
1. **Forgetting to advance the pointer** → infinite loop.
2. **Losing the next pointer** before reassigning it — always save `next = curr.next` first.
3. **Edge cases on head** — operations that work fine in the middle break when head is the target. The **dummy node** trick fixes this.
4. **Null pointer dereferences** — check `if curr is not None` before accessing `curr.next`.
5. **Off-by-one in two-pointer setups** — does fast start at head or head.next?

### Singly vs doubly linked

**Singly linked** — each node has one `next` pointer.
**Doubly linked** — each node has `next` AND `prev`. Costs 2x memory but enables O(1) deletion given just the node (you can wire prev.next = next directly).

**When you NEED doubly linked:** LRU cache, browser history, anything requiring O(1) bidirectional traversal. Otherwise singly is enough.


In [ ]:
# Node classes — singly and doubly
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

class DoublyListNode:
    def __init__(self, val=0, prev=None, next=None):
        self.val = val
        self.prev = prev
        self.next = next

# Test helpers — build/print/compare against arrays
def from_array(arr):
    # Build a singly linked list from a Python list. Returns head.
    dummy = ListNode()
    cur = dummy
    for v in arr:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

def to_array(head):
    # Convert a singly linked list to a Python list.
    out = []
    cur = head
    while cur:
        out.append(cur.val)
        cur = cur.next
    return out

assert to_array(from_array([])) == []
assert to_array(from_array([1, 2, 3])) == [1, 2, 3]
print("List helpers OK")


### The dummy node pattern — fixing head edge cases

When an operation might modify the head, instead of writing `if head is None` and `if target == head` special cases, prepend a dummy node:

```
dummy -> head -> a -> b -> c
```

Now every "real" node has a predecessor, and you return `dummy.next` at the end. **This pattern removes at least half the bugs in linked list code.** Use it for: merge, delete with condition, partition, swap pairs.


## 2. Core linked list operations

### 2.1 Reverse a linked list (LC 206)

Three pointers: `prev`, `curr`, `next`. Walk forward, redirecting each `curr.next` to point at `prev`.

```
Initial:   None  <-  1 -> 2 -> 3 -> None
Step 1:    None <- 1     2 -> 3 -> None  (rewire 1.next to None)
Step 2:    None <- 1 <- 2     3 -> None  (rewire 2.next to 1)
Step 3:    None <- 1 <- 2 <- 3            (rewire 3.next to 2)
```

**Time O(n), Space O(1) iterative; O(n) stack for recursive.**


In [ ]:
# Iterative — the workhorse
def reverse_list(head):
    prev = None
    curr = head
    while curr:
        nxt = curr.next     # 1. save next BEFORE we overwrite curr.next
        curr.next = prev    # 2. rewire to point backward
        prev = curr         # 3. advance prev
        curr = nxt          # 4. advance curr
    return prev             # prev ends up at the new head

# Recursive — elegant but uses O(n) stack
def reverse_list_rec(head):
    if not head or not head.next:
        return head                          # base case: empty or single node
    new_head = reverse_list_rec(head.next)   # reverse the rest first
    head.next.next = head                    # now make the old "next" point back to head
    head.next = None                         # head is now the new tail
    return new_head

# Tests
for arr in [[], [1], [1, 2], [1, 2, 3, 4, 5]]:
    assert to_array(reverse_list(from_array(arr))) == arr[::-1]
    assert to_array(reverse_list_rec(from_array(arr))) == arr[::-1]
print("reverse_list: OK (iterative and recursive)")


### 2.2 Find middle of a linked list (LC 876) — slow/fast pointers

**Trick:** advance `slow` one step and `fast` two steps. When `fast` reaches the end, `slow` is at the middle.

**Even length convention:** for 1->2->3->4, the "middle" can mean either 2 or 3 depending on problem definition. Using `while fast and fast.next` gives the **second middle** (3); using `while fast.next and fast.next.next` gives the **first middle** (2). Read the problem.

**Time O(n), Space O(1).**


In [ ]:
def middle_node(head):
    slow = fast = head
    while fast and fast.next:        # second middle on even length
        slow = slow.next
        fast = fast.next.next
    return slow

assert middle_node(from_array([1, 2, 3, 4, 5])).val == 3
assert middle_node(from_array([1, 2, 3, 4, 5, 6])).val == 4   # second middle
print("middle_node: OK")


### 2.3 Detect cycle (LC 141) — Floyd's tortoise and hare

Slow steps 1, fast steps 2. If there's a cycle, the gap closes by 1 each step → they meet inside the cycle. If `fast` hits null, no cycle.

**Why this works:** once both pointers are in the cycle, their relative position changes by 1 per iteration, so they must meet within (cycle length) iterations.

**Time O(n), Space O(1).** Beats the hash-set approach which is O(n) space.


In [ ]:
def has_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:                  # they met → cycle exists
            return True
    return False

# Cycle test
node1 = ListNode(1); node2 = ListNode(2); node3 = ListNode(3); node4 = ListNode(4)
node1.next = node2; node2.next = node3; node3.next = node4; node4.next = node2   # cycle to node2
assert has_cycle(node1) == True
assert has_cycle(from_array([1, 2, 3])) == False
print("has_cycle: OK")


### 2.4 Find cycle entry (LC 142)

After Floyd detects a meeting point inside the cycle, **reset slow to head** and advance both one step at a time. They meet at the cycle entry.

**Math behind it.** Let L = length to cycle start; let C = cycle length; let X = distance from cycle start to meeting point inside the cycle.
- Slow traveled L + X.
- Fast traveled 2(L + X), and also some multiple of C.
- So 2(L + X) - (L + X) = kC → L + X = kC → L = kC - X = (k-1)C + (C - X).

Translation: starting from head, walking L steps lands you at the cycle entry. Starting from the meeting point, walking (C - X) steps also lands at the cycle entry. So if both walk in lockstep, they meet at the entry. 

**Time O(n), Space O(1).**


In [ ]:
def detect_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:                  # met inside cycle
            slow = head
            while slow is not fast:
                slow = slow.next
                fast = fast.next
            return slow                   # cycle entry
    return None                           # no cycle

assert detect_cycle(node1) is node2       # entry of cycle is node2
assert detect_cycle(from_array([1, 2, 3])) is None
print("detect_cycle: OK")


### 2.5 Merge two sorted lists (LC 21)

Walk both lists with two pointers, picking the smaller value. **Use a dummy node** to avoid head edge cases.

**Time O(m + n), Space O(1)** if we splice in place.


In [ ]:
def merge_two_sorted(l1, l2):
    dummy = ListNode()                    # dummy head — saves edge-case logic
    tail = dummy
    while l1 and l2:
        if l1.val <= l2.val:
            tail.next = l1
            l1 = l1.next
        else:
            tail.next = l2
            l2 = l2.next
        tail = tail.next
    tail.next = l1 or l2                  # attach remainder
    return dummy.next

assert to_array(merge_two_sorted(from_array([1, 3, 5]), from_array([2, 4, 6]))) == [1, 2, 3, 4, 5, 6]
assert to_array(merge_two_sorted(from_array([]), from_array([0]))) == [0]
print("merge_two_sorted: OK")


### 2.6 Remove Nth node from end (LC 19) — one-pass with two pointers

Advance `fast` n steps first, then advance both until fast hits end. Use a dummy node so removing the head is uniform.

**Time O(n), Space O(1).**


In [ ]:
def remove_nth_from_end(head, n):
    dummy = ListNode(0, head)             # dummy so we can delete head uniformly
    fast = slow = dummy
    for _ in range(n):                    # advance fast by n
        fast = fast.next
    while fast.next:                      # advance both until fast.next is the tail
        fast = fast.next
        slow = slow.next
    slow.next = slow.next.next            # skip over the nth-from-end
    return dummy.next

assert to_array(remove_nth_from_end(from_array([1, 2, 3, 4, 5]), 2)) == [1, 2, 3, 5]
assert to_array(remove_nth_from_end(from_array([1]), 1)) == []
assert to_array(remove_nth_from_end(from_array([1, 2]), 1)) == [1]
print("remove_nth_from_end: OK")


## 3. Linked list patterns

### 3.1 Reverse in groups of K (LC 25)

For each group of k nodes, reverse them in place; if fewer than k remain, leave as-is (or reverse, depending on the variant).

**Algorithm:**
1. Count if there are at least k nodes ahead.
2. If yes, reverse k nodes; the original head of the group is now its tail. Recurse on the rest.
3. Wire the group tail to the result of the recursive call.

**Time O(n), Space O(n/k) recursion stack (or O(1) iteratively).**


In [ ]:
def reverse_k_group(head, k):
    # Check if we have at least k nodes ahead
    count = 0
    cur = head
    while cur and count < k:
        cur = cur.next
        count += 1
    if count < k:
        return head                       # leave remainder unreversed

    # Reverse the first k nodes
    prev = None
    cur = head
    for _ in range(k):
        nxt = cur.next
        cur.next = prev
        prev = cur
        cur = nxt
    # `head` is now the tail of this reversed group; `cur` is the start of the next group
    head.next = reverse_k_group(cur, k)
    return prev                           # new head of this group

assert to_array(reverse_k_group(from_array([1, 2, 3, 4, 5]), 2)) == [2, 1, 4, 3, 5]
assert to_array(reverse_k_group(from_array([1, 2, 3, 4, 5]), 3)) == [3, 2, 1, 4, 5]
print("reverse_k_group: OK")


### 3.2 Palindrome linked list (LC 234)

**Trick:** reverse the second half in place, then walk both halves with two pointers comparing values.

**Time O(n), Space O(1).** Beats the "copy to array" approach which is O(n) space.

(This is the exact problem in your reference.)


In [ ]:
def is_palindrome(head):
    # 1. Find middle
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    # 2. Reverse second half (from slow onward)
    prev = None
    while slow:
        nxt = slow.next
        slow.next = prev
        prev = slow
        slow = nxt
    # 3. Compare halves
    left, right = head, prev
    while right:                          # right is the shorter half (or equal)
        if left.val != right.val:
            return False
        left = left.next
        right = right.next
    return True

assert is_palindrome(from_array([1, 2, 2, 1])) == True
assert is_palindrome(from_array([1, 2, 3, 2, 1])) == True
assert is_palindrome(from_array([1, 2])) == False
assert is_palindrome(from_array([1])) == True
print("is_palindrome: OK")


### 3.3 Segregate odd and even nodes (or odd/even index — LC 328)

Two flavors:
- **By value** (your reference): split by even-value vs odd-value, then join.
- **By position** (LC 328): odd-indexed nodes first, then even-indexed.

Both follow the same template: maintain two separate sub-lists by walking once, then join.

**Time O(n), Space O(1).**


In [ ]:
# By POSITION (LC 328 — odd-indexed first, then even-indexed)
def odd_even_list_by_position(head):
    if not head or not head.next:
        return head
    odd = head                            # nodes at positions 1, 3, 5, ...
    even = head.next                      # nodes at positions 2, 4, 6, ...
    even_head = even                      # save to splice later
    while even and even.next:
        odd.next = even.next
        odd = odd.next
        even.next = odd.next
        even = even.next
    odd.next = even_head                  # splice even chain after odd chain
    return head

assert to_array(odd_even_list_by_position(from_array([1, 2, 3, 4, 5]))) == [1, 3, 5, 2, 4]
assert to_array(odd_even_list_by_position(from_array([2, 1, 3, 5, 6, 4, 7]))) == [2, 3, 6, 7, 1, 5, 4]
print("odd_even_list_by_position: OK")

# By VALUE (your reference)
def segregate_even_odd_by_value(head):
    even_head = even_tail = None
    odd_head = odd_tail = None
    cur = head
    while cur:
        nxt = cur.next
        cur.next = None                   # detach
        if cur.val % 2 == 0:
            if not even_head: even_head = even_tail = cur
            else: even_tail.next = cur; even_tail = cur
        else:
            if not odd_head: odd_head = odd_tail = cur
            else: odd_tail.next = cur; odd_tail = cur
        cur = nxt
    if even_tail:
        even_tail.next = odd_head
        return even_head
    return odd_head

assert to_array(segregate_even_odd_by_value(from_array([1, 2, 3, 4, 5]))) == [2, 4, 1, 3, 5]
print("segregate_even_odd_by_value: OK")


### 3.4 Clone a linked list with random pointers (LC 138)

Each node has a `next` pointer AND a `random` pointer to anywhere (or null). Deep copy the list in O(n), O(1) extra space (excluding output).

**The 3-pass trick** (your reference's algorithm):
1. **Interleave clones:** for every original node X, insert a clone X' right after it. So `X -> X'-> Y -> Y' -> ...`
2. **Wire random pointers:** for each X', its random is `X.random.next` (because `X.random.next` is the clone of whatever `X.random` was).
3. **Detach:** un-interleave the two lists.

**Time O(n), Space O(1).** Without this trick, you'd need an O(n) hashmap from original to clone.


In [ ]:
class RandomNode:
    def __init__(self, val=0, next=None, random=None):
        self.val = val
        self.next = next
        self.random = random

def clone_random_list(head):
    if not head:
        return None
    # Pass 1: interleave clones
    cur = head
    while cur:
        clone = RandomNode(cur.val, cur.next)    # next is the original's next
        cur.next = clone
        cur = clone.next
    # Pass 2: wire randoms
    cur = head
    while cur:
        if cur.random:
            cur.next.random = cur.random.next    # cur.random.next IS the clone of cur.random
        cur = cur.next.next
    # Pass 3: detach
    cur = head
    clone_head = head.next
    while cur:
        clone = cur.next
        cur.next = clone.next                    # restore original's next
        clone.next = clone.next.next if clone.next else None
        cur = cur.next
    return clone_head

# Build a list and verify
a = RandomNode(1); b = RandomNode(2); c = RandomNode(3)
a.next = b; b.next = c
a.random = c; b.random = a; c.random = b
cloned = clone_random_list(a)
# Verify clone preserves vals and random pointer values
vals, rands = [], []
cur = cloned
while cur:
    vals.append(cur.val)
    rands.append(cur.random.val if cur.random else None)
    cur = cur.next
assert vals == [1, 2, 3]
assert rands == [3, 1, 2]
# Verify it's actually a deep copy
assert cloned is not a
print("clone_random_list: OK — O(n), O(1) space")


### 3.5 LRU Cache (LC 146)

Hashmap + doubly linked list. The hashmap gives O(1) key lookup; the doubly linked list maintains usage order with O(1) move-to-front and O(1) eviction at tail.

**Pattern:** every `get` and `put` operation involves (a) hashmap lookup, (b) DLL node move to front. On eviction, remove DLL tail and corresponding hashmap key.

**Time O(1) per operation, Space O(capacity).**

**The trap.** `OrderedDict` from collections is the Pythonic shortcut — it gives you the same complexity with much less code. But interviewers often want you to write the hashmap+DLL by hand to prove you understand the underlying mechanics. Mention both.


In [ ]:
# Approach A: handcrafted hashmap + doubly linked list (interview-style)
class LRUCache:
    class Node:
        def __init__(self, key, val):
            self.key = key
            self.val = val
            self.prev = None
            self.next = None

    def __init__(self, capacity):
        self.cap = capacity
        self.map = {}                         # key → Node
        # Sentinel head/tail to avoid edge cases — head.next is MRU, tail.prev is LRU
        self.head = self.Node(0, 0)
        self.tail = self.Node(0, 0)
        self.head.next = self.tail
        self.tail.prev = self.head

    def _remove(self, node):                  # detach from list
        node.prev.next = node.next
        node.next.prev = node.prev

    def _add_to_front(self, node):            # insert just after head sentinel
        node.next = self.head.next
        node.prev = self.head
        self.head.next.prev = node
        self.head.next = node

    def get(self, key):
        if key not in self.map:
            return -1
        node = self.map[key]
        self._remove(node)
        self._add_to_front(node)              # move to front on access
        return node.val

    def put(self, key, value):
        if key in self.map:
            self._remove(self.map[key])
        node = self.Node(key, value)
        self.map[key] = node
        self._add_to_front(node)
        if len(self.map) > self.cap:
            lru = self.tail.prev               # LRU is just before tail sentinel
            self._remove(lru)
            del self.map[lru.key]

cache = LRUCache(2)
cache.put(1, 1); cache.put(2, 2)
assert cache.get(1) == 1                       # 1 → MRU; order is [1, 2]
cache.put(3, 3)                                # evicts 2
assert cache.get(2) == -1
cache.put(4, 4)                                # evicts 1
assert cache.get(1) == -1
assert cache.get(3) == 3
assert cache.get(4) == 4
print("LRUCache (handcrafted): OK — O(1) per op")


In [ ]:
# Approach B: the Pythonic shortcut using OrderedDict
from collections import OrderedDict

class LRUCachePythonic:
    def __init__(self, capacity):
        self.cap = capacity
        self.cache = OrderedDict()

    def get(self, key):
        if key not in self.cache:
            return -1
        self.cache.move_to_end(key)            # mark as MRU
        return self.cache[key]

    def put(self, key, value):
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.cap:
            self.cache.popitem(last=False)     # pop LRU from the front

c = LRUCachePythonic(2)
c.put(1, 1); c.put(2, 2)
assert c.get(1) == 1
c.put(3, 3)
assert c.get(2) == -1
print("LRUCachePythonic: OK")


### 3.6 Sort a linked list — merge sort (LC 148)

Merge sort fits linked lists naturally because we don't need random access. **Quicksort on linked lists is a nightmare** (no O(1) random-access pivot); merge sort wins.

**Time O(n log n), Space O(log n) recursion stack.**

The pattern: find middle → split → recurse on each half → merge with the function from §2.5.


In [ ]:
def sort_list(head):
    if not head or not head.next:
        return head
    # 1. Split into two halves (use first-middle to keep left-heavy on odd)
    slow, fast = head, head.next
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    right = slow.next
    slow.next = None                           # cut the list
    # 2. Recurse + merge
    return merge_two_sorted(sort_list(head), sort_list(right))

assert to_array(sort_list(from_array([4, 2, 1, 3]))) == [1, 2, 3, 4]
assert to_array(sort_list(from_array([-1, 5, 3, 4, 0]))) == [-1, 0, 3, 4, 5]
print("sort_list: OK — O(n log n)")


### 3.7 Add two numbers (LC 2 / 445)

Two lists represent numbers (LC 2: in reverse order, easy; LC 445: in forward order, requires reversing first or using a stack).

**Time O(max(m, n)), Space O(max(m, n)).** The carry-handling is the only tricky bit.


In [ ]:
def add_two_numbers_reverse(l1, l2):
    # LC 2: digits stored in reverse order, e.g. 342 = 2->4->3.
    dummy = ListNode()
    cur = dummy
    carry = 0
    while l1 or l2 or carry:
        v = carry
        if l1: v += l1.val; l1 = l1.next
        if l2: v += l2.val; l2 = l2.next
        carry, digit = divmod(v, 10)
        cur.next = ListNode(digit)
        cur = cur.next
    return dummy.next

assert to_array(add_two_numbers_reverse(from_array([2, 4, 3]), from_array([5, 6, 4]))) == [7, 0, 8]
assert to_array(add_two_numbers_reverse(from_array([9, 9]), from_array([1]))) == [0, 0, 1]
print("add_two_numbers_reverse: OK")


### Linked list practice problems

| Concept | LeetCode |
|---|---|
| Reverse Linked List | LC 206 |
| Reverse Linked List II (sublist) | LC 92 |
| Reverse Nodes in K-Group | LC 25 |
| Middle of the Linked List | LC 876 |
| Linked List Cycle | LC 141 |
| Linked List Cycle II | LC 142 |
| Merge Two Sorted Lists | LC 21 |
| Merge K Sorted Lists | LC 23 (covered in heap notebook) |
| Remove Nth From End | LC 19 |
| Palindrome Linked List | LC 234 |
| Odd Even Linked List | LC 328 |
| Copy List with Random Pointer | LC 138 |
| LRU Cache | LC 146 |
| LFU Cache | LC 460 |
| Sort List | LC 148 |
| Add Two Numbers | LC 2 |
| Add Two Numbers II | LC 445 |
| Intersection of Two Linked Lists | LC 160 |
| Swap Nodes in Pairs | LC 24 |
| Rotate List | LC 61 |
| Flatten a Multilevel Doubly Linked List | LC 430 |


# Part 2 — Backtracking

## 4. What is backtracking?

Backtracking is **DFS with state restoration**. You build a candidate solution incrementally; whenever you hit a dead end or invalid state, you **undo the last step ("back" track)** and try the next option.

The key insight: instead of generating every possible candidate and filtering, you **prune as soon as the partial solution becomes invalid**. This converts what looks like a brute-force exponential search into something often tractable.

### The decision tree picture

Think of every choice as a node in a tree. The root is the empty solution; each branch is "what choice do I make next?"; leaves are complete solutions. Backtracking is a DFS traversal of this tree with **pruning** — when a subtree can't contain a valid solution, skip it entirely.

```
Build subsets of [1,2,3]:
                       []
              /         |        \
           [1]         [2]       [3]
          /    \        |
       [1,2]  [1,3]   [2,3]
         |
      [1,2,3]
```
Each leaf is a subset. The tree has 2^n leaves — backtracking visits them in DFS order.

### The universal backtracking template

Memorize this. **All 20+ backtracking problems are variants:**

```python
def backtrack(state):
    if is_solution(state):
        record(state)              # save a copy — state will be mutated
        return                     # (or continue if a solution can be extended)
    for choice in choices(state):
        if not valid(choice, state):
            continue               # prune
        apply(choice, state)       # make the choice
        backtrack(state)           # recurse
        undo(choice, state)        # ← THE BACKTRACK: undo to try next
```

The variation between problems is just:
- **What's the state?** (current subset, current permutation, current board, etc.)
- **What's a complete solution?** (length == k, used all elements, all cells filled, etc.)
- **What are the choices?** (next element, next cell value, next position to place a queen)
- **What's valid?** (no duplicates, no conflict with existing placements)

### Why the "undo" step matters

Without explicit undo, the state would be permanently mutated and the next sibling in the decision tree would explore from the wrong state.

Some problems sidestep this by **passing immutable state** (e.g., a string concatenation rather than appending to a list). But the canonical mutable-state-plus-undo pattern is more efficient and what you'll write 90% of the time.

### Complexity

Backtracking is **typically exponential** — you're walking a decision tree. The exact complexity depends on the branching factor and pruning. Examples:
- All subsets: O(n · 2ⁿ) — 2ⁿ subsets, O(n) to copy each.
- All permutations: O(n · n!) — n! perms, O(n) to copy.
- N-Queens: O(n!) decision tree, heavily pruned in practice.
- Sudoku: 9^81 worst case, but pruning makes it solvable in milliseconds.

**Don't memorize complexities — compute them from the recurrence:** branching factor × depth.


## 5. The Big Three — subsets, permutations, combinations

These three problems are the canonical backtracking templates. Master them and you master the topic.

### 5.1 Subsets (LC 78)

Generate the power set: every possible subset.

**Decision at each level:** "do I include the next element or not?" Branching factor 2. Total subsets 2ⁿ.

**Time O(n · 2ⁿ), Space O(n) recursion + O(2ⁿ · n) output.**


In [ ]:
def subsets(nums):
    out = []

    def backtrack(start, path):
        out.append(path[:])                    # snapshot — every state is a valid subset
        for i in range(start, len(nums)):
            path.append(nums[i])               # choose
            backtrack(i + 1, path)             # explore
            path.pop()                         # un-choose (backtrack)

    backtrack(0, [])
    return out

result = subsets([1, 2, 3])
assert len(result) == 8                        # 2^3
expected = [[], [1], [1,2], [1,2,3], [1,3], [2], [2,3], [3]]
assert sorted(map(tuple, result)) == sorted(map(tuple, expected))
print("subsets: OK — O(n · 2^n)")


### 5.2 Subsets with duplicates (LC 90)

Same as above, but the input can contain duplicates and we want **unique** subsets.

**Trick:** sort the input. Then, within a single level of recursion, **skip duplicates of an already-chosen element.**

```
nums = [1, 2, 2]
       []
    /  |  \
  [1] [2]  ❌ skip second 2 at top level
   /\   \
  [1,2] [1,2,2]
```


In [ ]:
def subsets_with_dup(nums):
    nums.sort()
    out = []

    def backtrack(start, path):
        out.append(path[:])
        for i in range(start, len(nums)):
            # Skip duplicates within this recursion level
            if i > start and nums[i] == nums[i-1]:
                continue
            path.append(nums[i])
            backtrack(i + 1, path)
            path.pop()

    backtrack(0, [])
    return out

result = subsets_with_dup([1, 2, 2])
expected = [[], [1], [1,2], [1,2,2], [2], [2,2]]
assert sorted(map(tuple, result)) == sorted(map(tuple, expected))
print("subsets_with_dup: OK")


### 5.3 Permutations (LC 46)

**Decision at each level:** pick any unused element. Branching factor at depth d is (n - d). Total n!.

**Two ways to track "used":**
1. **`used` boolean array** — explicit.
2. **In-place swap** — swap the chosen element into position d, then recurse on d+1.

Both are O(n · n!) time. Swap version uses less auxiliary memory.

**Time O(n · n!), Space O(n) recursion.**


In [ ]:
# Version A: used array (clearer)
def permute(nums):
    out = []
    used = [False] * len(nums)

    def backtrack(path):
        if len(path) == len(nums):
            out.append(path[:])
            return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True
            path.append(nums[i])
            backtrack(path)
            path.pop()
            used[i] = False

    backtrack([])
    return out

result = permute([1, 2, 3])
assert len(result) == 6                        # 3!
assert sorted(map(tuple, result)) == sorted([(1,2,3),(1,3,2),(2,1,3),(2,3,1),(3,1,2),(3,2,1)])
print("permute: OK — O(n · n!)")

# Version B: in-place swap
def permute_swap(nums):
    out = []
    nums = nums[:]                             # copy so we don't mutate caller's list

    def backtrack(start):
        if start == len(nums):
            out.append(nums[:])
            return
        for i in range(start, len(nums)):
            nums[start], nums[i] = nums[i], nums[start]   # swap into position
            backtrack(start + 1)
            nums[start], nums[i] = nums[i], nums[start]   # swap back

    backtrack(0)
    return out

assert len(permute_swap([1, 2, 3])) == 6
print("permute_swap: OK")


### 5.4 Permutations with duplicates (LC 47)

Sort + within-level dedup, similar to subsets-with-dup but on the `used` array.


In [ ]:
def permute_unique(nums):
    nums.sort()
    out = []
    used = [False] * len(nums)

    def backtrack(path):
        if len(path) == len(nums):
            out.append(path[:])
            return
        for i in range(len(nums)):
            if used[i]:
                continue
            # Skip duplicates: if previous identical was NOT used in this branch, skip
            if i > 0 and nums[i] == nums[i-1] and not used[i-1]:
                continue
            used[i] = True
            path.append(nums[i])
            backtrack(path)
            path.pop()
            used[i] = False

    backtrack([])
    return out

result = permute_unique([1, 1, 2])
expected = [[1,1,2], [1,2,1], [2,1,1]]
assert sorted(map(tuple, result)) == sorted(map(tuple, expected))
print("permute_unique: OK")


### 5.5 Combinations (LC 77 / 39 / 40)

**Combinations** (LC 77): pick k elements from [1..n], order doesn't matter.

Branching factor at depth d is (n - last_chosen) (i.e., elements still available).

**Combination Sum** (LC 39): pick from candidates (with repetition) that sum to target.

**Combination Sum II** (LC 40): same but each element used at most once + duplicates in input.

These all follow the same "start index, pick or skip" template. The differences are in (a) target check (b) `i+1` vs `i` for the recursive call (with vs without reuse).

**Time:** depends. LC 77 is O(C(n,k) · k). LC 39 worst-case exponential.


In [ ]:
# LC 77: combinations C(n, k)
def combinations(n, k):
    out = []

    def backtrack(start, path):
        if len(path) == k:
            out.append(path[:])
            return
        # Pruning: if we can't possibly reach k more elements, give up
        for i in range(start, n + 1):
            if len(path) + (n - i + 1) < k:    # not enough remaining
                break
            path.append(i)
            backtrack(i + 1, path)
            path.pop()

    backtrack(1, [])
    return out

result = combinations(4, 2)
expected = [[1,2], [1,3], [1,4], [2,3], [2,4], [3,4]]
assert sorted(map(tuple, result)) == sorted(map(tuple, expected))
print("combinations: OK")

# LC 39: combination sum (with repetition)
def combination_sum(candidates, target):
    candidates.sort()
    out = []

    def backtrack(start, path, total):
        if total == target:
            out.append(path[:])
            return
        for i in range(start, len(candidates)):
            if total + candidates[i] > target:
                break                          # sorted → no point continuing
            path.append(candidates[i])
            backtrack(i, path, total + candidates[i])   # i (not i+1) → can reuse
            path.pop()

    backtrack(0, [], 0)
    return out

result = combination_sum([2, 3, 6, 7], 7)
expected = [[2, 2, 3], [7]]
assert sorted(map(tuple, result)) == sorted(map(tuple, expected))
print("combination_sum: OK")


## 6. Other backtracking families

### 6.1 N-Queens (LC 51 / 52)

Place N queens on an N×N board such that no two attack each other (no shared row, column, or diagonal).

**State:** which row we're filling. Place one queen per row. Branching factor at each row = N (one column choice), heavily pruned by attack checks.

**Pruning trick:** maintain three sets — `cols`, `diag1` (r+c is constant on \ diagonals), `diag2` (r-c is constant on / diagonals). Each check is O(1).

**Time O(N!) worst case; in practice much faster due to pruning.**


In [ ]:
def solve_n_queens(n):
    out = []
    cols = set()
    diag1 = set()                              # r + c (\\ diagonal)
    diag2 = set()                              # r - c (/ diagonal)
    queens = []                                # column placement per row

    def backtrack(r):
        if r == n:
            board = []
            for c in queens:
                row = '.' * c + 'Q' + '.' * (n - c - 1)
                board.append(row)
            out.append(board)
            return
        for c in range(n):
            if c in cols or (r + c) in diag1 or (r - c) in diag2:
                continue                       # pruned: attacked
            cols.add(c); diag1.add(r+c); diag2.add(r-c); queens.append(c)
            backtrack(r + 1)
            cols.remove(c); diag1.remove(r+c); diag2.remove(r-c); queens.pop()

    backtrack(0)
    return out

assert len(solve_n_queens(4)) == 2             # there are 2 solutions for n=4
assert len(solve_n_queens(5)) == 10
assert len(solve_n_queens(8)) == 92
print("solve_n_queens: OK")


### 6.2 Word Search (LC 79)

Given a grid of characters, find if a word can be spelled by walking adjacent cells (no reuse).

**State:** current cell + index into the target word.
**Choices at each step:** the four neighbors.
**Backtrack:** mark cell as visited before recursing; unmark on return.

**Trick to avoid a visited set:** temporarily mutate the cell to a sentinel character (`#`); restore on backtrack. Saves O(R*C) space.

**Time O(R · C · 4^L)** where L = word length, but heavily pruned in practice.


In [ ]:
def word_search(board, word):
    R, C = len(board), len(board[0])

    def dfs(r, c, i):
        if i == len(word):
            return True
        if r < 0 or r >= R or c < 0 or c >= C or board[r][c] != word[i]:
            return False
        # Mark visited by temporarily replacing the char
        saved = board[r][c]
        board[r][c] = '#'
        found = (dfs(r+1, c, i+1) or dfs(r-1, c, i+1)
                 or dfs(r, c+1, i+1) or dfs(r, c-1, i+1))
        board[r][c] = saved                    # restore on backtrack
        return found

    for r in range(R):
        for c in range(C):
            if dfs(r, c, 0):
                return True
    return False

board = [["A","B","C","E"],
         ["S","F","C","S"],
         ["A","D","E","E"]]
assert word_search([row[:] for row in board], "ABCCED") == True
assert word_search([row[:] for row in board], "SEE") == True
assert word_search([row[:] for row in board], "ABCB") == False
print("word_search: OK")


### 6.3 Sudoku Solver (LC 37)

Fill a 9×9 grid such that every row, column, and 3×3 box contains 1-9.

**State:** the partial board.
**Choices:** for each empty cell, try digits 1-9.
**Valid?** Digit not already in the same row, column, or 3x3 box.

**Optimization:** precompute the sets of used digits per row/col/box → O(1) validity check.

**Time:** exponential worst case, but the constraint propagation makes real Sudokus solve in milliseconds.

### 6.4 Palindrome Partitioning (LC 131)

Partition a string into substrings such that every substring is a palindrome. Return all partitions.

**State:** start index in the string.
**Choices:** every prefix from start that is a palindrome.

**Time O(n · 2ⁿ)** — n positions, 2ⁿ partition options worst case.


In [ ]:
def palindrome_partition(s):
    out = []

    def is_palindrome(left, right):
        while left < right:
            if s[left] != s[right]: return False
            left += 1; right -= 1
        return True

    def backtrack(start, path):
        if start == len(s):
            out.append(path[:])
            return
        for end in range(start, len(s)):
            if is_palindrome(start, end):
                path.append(s[start:end+1])
                backtrack(end + 1, path)
                path.pop()

    backtrack(0, [])
    return out

result = palindrome_partition("aab")
expected = [["a","a","b"], ["aa","b"]]
assert sorted(map(tuple, result)) == sorted(map(tuple, expected))
print("palindrome_partition: OK")


### 6.5 Letter Combinations of a Phone Number (LC 17)

Each digit (2-9) maps to 3-4 letters. Generate all letter combinations corresponding to a digit string.

A small, clean backtracking warm-up.


In [ ]:
def letter_combinations(digits):
    if not digits: return []
    mp = {'2':'abc','3':'def','4':'ghi','5':'jkl','6':'mno','7':'pqrs','8':'tuv','9':'wxyz'}
    out = []

    def backtrack(i, path):
        if i == len(digits):
            out.append(''.join(path))
            return
        for ch in mp[digits[i]]:
            path.append(ch)
            backtrack(i + 1, path)
            path.pop()

    backtrack(0, [])
    return out

result = letter_combinations("23")
expected = ["ad","ae","af","bd","be","bf","cd","ce","cf"]
assert sorted(result) == sorted(expected)
print("letter_combinations: OK")


### Backtracking practice problems

| Concept | LeetCode |
|---|---|
| Subsets | LC 78 |
| Subsets II (duplicates) | LC 90 |
| Permutations | LC 46 |
| Permutations II (duplicates) | LC 47 |
| Combinations | LC 77 |
| Combination Sum | LC 39 |
| Combination Sum II | LC 40 |
| Combination Sum III | LC 216 |
| Letter Combinations of a Phone Number | LC 17 |
| Word Search | LC 79 |
| Word Search II (with Trie) | LC 212 |
| N-Queens | LC 51 |
| N-Queens II (count only) | LC 52 |
| Sudoku Solver | LC 37 |
| Restore IP Addresses | LC 93 |
| Generate Parentheses | LC 22 |
| Palindrome Partitioning | LC 131 |
| Partition to K Equal Sum Subsets | LC 698 |


# Part 3 — Greedy Algorithms

## 7. What is greedy?

A **greedy algorithm** makes the locally optimal choice at each step, hoping it leads to a globally optimal solution.

The hard part is **proving** (or convincingly arguing) that the greedy choice IS globally optimal. Many greedy approaches look right and are wrong. **The art is recognizing when greedy works.**

### Greedy choice property + Optimal substructure

A problem is greedy-solvable iff:
1. **Greedy choice property:** a globally optimal solution can be reached by making locally optimal choices.
2. **Optimal substructure:** the optimal answer to the whole problem contains optimal answers to its subproblems.

Both DP and greedy require optimal substructure. The difference: **DP tries all options and picks the best**; **greedy commits to one option without revisiting**.

### The exchange argument

The standard proof technique for greedy correctness:
1. Assume the greedy solution G differs from some optimal solution O at some step.
2. Show that you can "exchange" O's choice for G's choice without making the solution worse.
3. Therefore G is at least as good as O. ⊡

When you can construct an exchange argument, greedy works. When you can't, try DP.

### When greedy FAILS — and what to do instead

Greedy fails when local optimal ≠ global optimal. Classic example: **coin change with arbitrary denominations**. With coins [1, 3, 4] and amount 6, greedy picks 4+1+1=3 coins; the optimum is 3+3=2 coins. The greedy choice (largest coin first) is locally optimal but globally suboptimal.

The fix for failed-greedy problems: **dynamic programming**. The coin change problem is solved with DP in O(amount · #coins).

**Rule of thumb:** if you can't prove the exchange argument quickly, reach for DP.

### Common greedy patterns

| Pattern | Example |
|---|---|
| **Sort + sweep** | Merge intervals, meeting rooms |
| **Pick the best available** | Activity selection (earliest finish), Huffman (smallest two trees) |
| **Frontier expansion** | Jump game, gas station |
| **Priority queue** | Task scheduler, IPO, reorganize string (see heap notebook) |


## 8. Classic greedy problems

### 8.1 Activity Selection / Non-overlapping Intervals (LC 435)

Given intervals, find the minimum number of intervals to remove so the rest don't overlap.

**Greedy choice:** sort by **end time**, then sweep. Always keep the interval that ends earliest among non-overlapping candidates.

**Exchange argument intuition:** if you skip the earliest-ending interval, you could swap it in without losing anything — it leaves the most room for future intervals.

**Time O(n log n) for sort, O(n) sweep.**


In [ ]:
def erase_overlap_intervals(intervals):
    intervals.sort(key=lambda x: x[1])         # sort by END time — the greedy key
    end = float('-inf')
    kept = 0
    for start, finish in intervals:
        if start >= end:                       # no overlap with last kept
            kept += 1
            end = finish
    return len(intervals) - kept

assert erase_overlap_intervals([[1,2],[2,3],[3,4],[1,3]]) == 1
assert erase_overlap_intervals([[1,2],[1,2],[1,2]]) == 2
assert erase_overlap_intervals([[1,2],[2,3]]) == 0
print("erase_overlap_intervals: OK — sort by end")


### 8.2 Jump Game (LC 55)

Each `nums[i]` = max jump length from i. Can you reach the last index?

**Greedy:** track the rightmost reachable index. If at index i you've already failed to reach i, return False. Else update.

**Time O(n), Space O(1).**


In [ ]:
def can_jump(nums):
    farthest = 0
    for i, v in enumerate(nums):
        if i > farthest: return False          # can't reach this index
        farthest = max(farthest, i + v)
        if farthest >= len(nums) - 1: return True
    return True

assert can_jump([2, 3, 1, 1, 4]) == True
assert can_jump([3, 2, 1, 0, 4]) == False
print("can_jump: OK — greedy frontier")


### 8.3 Jump Game II — minimum jumps (LC 45)

Min jumps to reach the last index.

**Greedy:** BFS layered. Each "layer" is a jump. Track the **farthest** reachable within the current jump; when you exhaust the current jump's reach, increment jumps and update the reach.

**Time O(n), Space O(1).**


In [ ]:
def jump(nums):
    jumps = 0
    current_end = 0                            # reach with `jumps` jumps
    farthest = 0                               # reach with `jumps + 1` jumps
    for i in range(len(nums) - 1):
        farthest = max(farthest, i + nums[i])
        if i == current_end:                   # used up current jump
            jumps += 1
            current_end = farthest
    return jumps

assert jump([2, 3, 1, 1, 4]) == 2
assert jump([2, 3, 0, 1, 4]) == 2
print("jump: OK")


### 8.4 Gas Station (LC 134)

Circular tour problem (same as your reference's `first_circular_tour`). Find a starting station such that you can complete the loop.

**Greedy insight:** if you run out of gas going from i to j, then no station between i and j can be a valid start. Reset to j+1.

**Why?** If you start at i with 0 tank and run out somewhere before j, you'd run out worse starting at any station between i and j (you'd have less gas accumulated).

**Time O(n), Space O(1).** Covered in the queues/heap notebook too.


In [ ]:
def gas_station(gas, cost):
    total_surplus = 0
    tank = 0
    start = 0
    for i in range(len(gas)):
        gain = gas[i] - cost[i]
        tank += gain
        total_surplus += gain
        if tank < 0:
            start = i + 1
            tank = 0
    return start if total_surplus >= 0 else -1

assert gas_station([1,2,3,4,5], [3,4,5,1,2]) == 3
assert gas_station([2,3,4], [3,4,3]) == -1
print("gas_station: OK")


### 8.5 Minimum Number of Arrows (LC 452)

Balloons on a number line, each interval [x_start, x_end]. Find min arrows to burst all balloons (an arrow at position p bursts every balloon whose interval contains p).

**Greedy:** sort by **end** position. Shoot an arrow at the end of the leftmost balloon; it bursts every balloon starting before this position. Continue from the next un-burst balloon.

**Time O(n log n).**


In [ ]:
def find_min_arrows(points):
    if not points: return 0
    points.sort(key=lambda x: x[1])             # sort by end
    arrows = 1
    end = points[0][1]
    for start, finish in points[1:]:
        if start > end:                         # need a new arrow
            arrows += 1
            end = finish
    return arrows

assert find_min_arrows([[10,16],[2,8],[1,6],[7,12]]) == 2
assert find_min_arrows([[1,2],[2,3],[3,4],[4,5]]) == 2
print("find_min_arrows: OK")


### 8.6 Assign Cookies (LC 455)

n kids with greed factors g[i]; m cookies of sizes s[j]. Assign at most one cookie per kid so the cookie size ≥ greed factor. Maximize satisfied kids.

**Greedy:** sort both arrays. Assign the smallest sufficient cookie to the least greedy child. Two-pointer sweep.

**Time O(n log n + m log m).**


In [ ]:
def find_content_children(g, s):
    g.sort(); s.sort()
    i = j = 0
    while i < len(g) and j < len(s):
        if s[j] >= g[i]:                        # cookie satisfies child
            i += 1
        j += 1                                   # move to next cookie either way
    return i

assert find_content_children([1, 2, 3], [1, 1]) == 1
assert find_content_children([1, 2], [1, 2, 3]) == 2
print("find_content_children: OK")


## 9. Greedy reasoning — when can you prove it works?

### Practice: a problem where greedy looks right but is wrong

**Coin Change.** With denominations [1, 3, 4] and amount 6:
- Greedy (largest first): 4 + 1 + 1 = 3 coins.
- Optimal: 3 + 3 = 2 coins.

Greedy fails because **picking 4 prevents the better 3+3 combination**. The locally-best choice doesn't extend to global optimum.

**The fix:** DP (covered in the recursion+DP notebook §10).

### A problem where greedy WORKS — and you should be able to argue why

**Earliest-deadline-first scheduling.** N jobs with deadlines and unit duration; maximize jobs completed on time.

Greedy: sort by deadline; schedule each as late as possible without exceeding its deadline (using a max-heap of currently-feasible jobs).

**Exchange argument:** if an optimal schedule does job A after job B but A has an earlier deadline, swap them — both still meet their deadlines, so the swap doesn't hurt.

### How to decide greedy vs DP in an interview

1. **Sketch the brute force.** Is it "try every choice"? That's DP territory.
2. **Can you sort by something and sweep?** Almost certainly greedy.
3. **Does the locally-best choice depend on the future?** Probably DP.
4. **Can you construct a small counterexample to your greedy?** If yes, ditch it for DP.

### Greedy practice problems

| Concept | LeetCode |
|---|---|
| Jump Game | LC 55 |
| Jump Game II | LC 45 |
| Gas Station | LC 134 |
| Assign Cookies | LC 455 |
| Lemonade Change | LC 860 |
| Non-overlapping Intervals | LC 435 |
| Minimum Number of Arrows | LC 452 |
| Best Time to Buy and Sell Stock II | LC 122 |
| Task Scheduler | LC 621 |
| Reorganize String | LC 767 |
| Two City Scheduling | LC 1029 |
| Boats to Save People | LC 881 |
| Partition Labels | LC 763 |
| Hand of Straights | LC 846 |
| Candy | LC 135 |
| Wiggle Subsequence | LC 376 |
| Queue Reconstruction by Height | LC 406 |


# Part 4 — Mathematics for Interviews

## 10. GCD, LCM, and the Euclidean algorithm

### 10.1 GCD via Euclidean algorithm

The **greatest common divisor** (GCD) of a and b is the largest integer dividing both. The Euclidean algorithm computes it in O(log(min(a, b))).

**The key identity:** `gcd(a, b) = gcd(b, a mod b)`.

**Proof (the same one in your reference, made rigorous):**

Let g = gcd(a, b). Write a = g·x, b = g·y with gcd(x, y) = 1 (otherwise g wouldn't be the GREATEST).

`a mod b = a - q·b = g·x - q·g·y = g·(x - q·y)`.

So g divides both b (= g·y) and a mod b (= g(x - qy)). Therefore g is a common divisor of b and (a mod b), so g ≤ gcd(b, a mod b).

Conversely, any common divisor of b and (a mod b) divides `b * q + (a mod b) = a`, so it's also a common divisor of a and b, hence ≤ g.

Therefore `gcd(a, b) = gcd(b, a mod b)`. ⊡

Using `a mod b` instead of repeated `a - b` is just an optimization — it skips many subtractions in one step.

**Time:** O(log(min(a, b))) — each step at least halves the larger number on average (Lamé's theorem).


In [ ]:
def gcd(a, b):
    while b:
        a, b = b, a % b
    return a

# Recursive version (same complexity, but stack-bound)
def gcd_rec(a, b):
    return a if b == 0 else gcd_rec(b, a % b)

assert gcd(10, 5) == 5
assert gcd(15, 20) == 5
assert gcd(100, 75) == 25
assert gcd(7, 13) == 1                          # coprime
assert gcd(0, 5) == 5                           # edge: gcd(0, x) = x
print("gcd: OK")

# Python has this in the standard library
import math
assert math.gcd(15, 20) == 5
print("math.gcd: also available")


### 10.2 LCM

`lcm(a, b) = a * b / gcd(a, b)`.

**Why?** Every prime p that divides either a or b contributes max(power in a, power in b) to lcm and min(...) to gcd. Their product is the sum, which equals the powers in `a * b`. So `lcm * gcd = a * b`.

**Careful:** compute `a // gcd(a, b) * b` instead of `a * b // gcd(a, b)` to avoid overflow in fixed-width languages. Python handles big ints natively but burn the habit in.


In [ ]:
def lcm(a, b):
    return a // gcd(a, b) * b                   # safer multiplication order

assert lcm(4, 6) == 12
assert lcm(7, 13) == 91                          # coprime → lcm = product
print("lcm: OK")


### 10.3 Extended Euclidean (mention)

Computes x, y such that `a*x + b*y = gcd(a, b)`. Used for:
- Modular inverse (solving `a*x ≡ 1 mod m`)
- Chinese Remainder Theorem
- Diophantine equations

Worth recognizing the name; rarely required to implement in standard interviews.

## 11. Primality, prime factorization, sieve

### 11.1 Check if a number is prime — O(√n)

A prime > 3 has form 6k ± 1. We can skip even numbers and multiples of 3.

**Naive:** check divisibility by 2, 3, 4, ..., n-1. O(n).

**Better:** check divisibility by 2, 3, 5, 7, ..., up to √n. **Why √n?** If n has a divisor d > √n, then n/d < √n must also be a divisor.

**Time O(√n).**


In [ ]:
def is_prime(n):
    if n < 2: return False
    if n < 4: return True                       # 2, 3 are prime
    if n % 2 == 0 or n % 3 == 0: return False
    i = 5
    while i * i <= n:                           # while i² ≤ n (equivalent to i ≤ √n)
        if n % i == 0 or n % (i + 2) == 0:      # check 6k-1 and 6k+1
            return False
        i += 6
    return True

assert is_prime(2) == True
assert is_prime(7) == True
assert is_prime(1) == False
assert is_prime(15) == False
assert is_prime(97) == True
assert is_prime(100) == False
assert is_prime(2**31 - 1) == True              # Mersenne prime
print("is_prime: OK — O(sqrt(n))")


### 11.2 Prime factorization — O(√n)

Repeatedly divide by primes from 2 upward.

**Trick.** Once you've divided out all 2s, only odd divisors remain. Once you've handled 2, you can skip evens. And once i exceeds √n of the **remaining** number, what's left (if > 1) must itself be a prime.

**Time O(√n).**


In [ ]:
def prime_factors(n):
    factors = []
    # Divide out 2s
    while n % 2 == 0:
        factors.append(2)
        n //= 2
    # Now n is odd; try odd divisors
    i = 3
    while i * i <= n:
        while n % i == 0:
            factors.append(i)
            n //= i
        i += 2
    # If n > 1 here, it's itself a prime
    if n > 1:
        factors.append(n)
    return factors

assert prime_factors(12) == [2, 2, 3]
assert prime_factors(45) == [3, 3, 5]
assert prime_factors(1) == []
assert prime_factors(97) == [97]
print("prime_factors: OK")


### 11.3 Sieve of Eratosthenes — all primes up to n in O(n log log n)

Mark composites by repeated multiples. Start from 2; mark 4, 6, 8, ... as composite. Then move to 3; mark 9, 12, 15, .... Then 5; ....

**Optimization:** start marking from `i*i` (smaller multiples already marked by smaller primes). Stop the outer loop at √n.

**Time:** O(n log log n), which is **essentially linear** for all interview-scale inputs.


In [ ]:
def sieve(n):
    # Return list of all primes up to n inclusive.
    if n < 2: return []
    is_p = [True] * (n + 1)
    is_p[0] = is_p[1] = False
    for i in range(2, int(n**0.5) + 1):
        if is_p[i]:
            # Mark multiples starting from i*i
            for j in range(i*i, n + 1, i):
                is_p[j] = False
    return [i for i, p in enumerate(is_p) if p]

assert sieve(2) == [2]
assert sieve(10) == [2, 3, 5, 7]
assert sieve(20) == [2, 3, 5, 7, 11, 13, 17, 19]
assert len(sieve(100)) == 25                    # 25 primes under 100
print("sieve: OK")


## 12. Fast exponentiation and modular arithmetic

### 12.1 Fast power — O(log n)

`x^n` naive is O(n). Using the identity `x^n = (x^(n/2))² · x^(n mod 2)`, we get O(log n).

**Recursive:**
```
x^0 = 1
x^n = (x^(n/2))²              if n even
x^n = x · (x^(n//2))²          if n odd
```

**Iterative (binary exponentiation):** walk the bits of n from LSB to MSB; whenever a bit is 1, multiply result by the current power-of-x; square x each step.

(Your reference has both; the iterative version is what we ship.)

**Time O(log n), Space O(1) iterative, O(log n) recursive.**


In [ ]:
def fast_pow(x, n):
    if n < 0:
        return 1 / fast_pow(x, -n)
    result = 1
    while n > 0:
        if n & 1:                                # if current bit is 1
            result *= x
        x *= x                                   # square the base
        n >>= 1                                  # next bit
    return result

assert fast_pow(2, 10) == 1024
assert fast_pow(3, 4) == 81
assert fast_pow(2, 0) == 1
assert fast_pow(2, -2) == 0.25
print("fast_pow: OK — O(log n)")


### 12.2 Modular fast power

For huge results, we want `x^n mod m`. Same algorithm, just `% m` at each multiply.

**Application:** every cryptographic system in the world. Also any "return answer mod 10^9 + 7" LeetCode problem.

Python has this built in: `pow(x, n, m)` computes x^n mod m in O(log n).


In [ ]:
def mod_pow(x, n, m):
    result = 1
    x %= m
    while n > 0:
        if n & 1:
            result = (result * x) % m
        x = (x * x) % m
        n >>= 1
    return result

assert mod_pow(2, 10, 1000) == 24                # 1024 mod 1000
assert mod_pow(3, 100, 7) == 4
# Python built-in
assert pow(2, 10, 1000) == 24
assert pow(3, 100, 7) == 4
print("mod_pow: OK — and pow(x, n, m) is built in")


### 12.3 Modular arithmetic rules

For modulus m, the following hold:
- `(a + b) % m == ((a % m) + (b % m)) % m`
- `(a - b) % m == ((a % m) - (b % m) + m) % m`  (add m to handle negative results)
- `(a * b) % m == ((a % m) * (b % m)) % m`
- Division is **NOT** just division — you need modular inverse: `(a / b) % m == (a * inv(b)) % m` where `inv(b) * b ≡ 1 mod m`.

**Modular inverse (when m is prime, by Fermat's little theorem):** `inv(b) = pow(b, m-2, m)`.

This is essential for any combinatorics problem with modular constraints (binomial coefficients mod p, etc.).

### 12.4 Counting digits / digit sum / reverse digits

Common micro-problems involving extracting digits via `% 10` and `// 10`.


In [ ]:
# Count digits in O(log n)
def count_digits(n):
    if n == 0: return 1
    n = abs(n)
    count = 0
    while n:
        count += 1
        n //= 10
    return count

# Sum of digits
def digit_sum(n):
    n = abs(n)
    total = 0
    while n:
        total += n % 10
        n //= 10
    return total

# Reverse digits (LC 7) — careful with sign and overflow in fixed-width
def reverse_int(n):
    sign = -1 if n < 0 else 1
    n = abs(n)
    rev = 0
    while n:
        rev = rev * 10 + n % 10
        n //= 10
    return sign * rev

assert count_digits(0) == 1
assert count_digits(12345) == 5
assert digit_sum(123) == 6
assert reverse_int(12345) == 54321
assert reverse_int(-120) == -21
print("digit ops: OK")


### 12.5 Combinatorics — counting

In Python, `math.comb(n, k)` and `math.perm(n, k)` give you binomial coefficients and permutations directly.

**Manually:**
- `n! = math.factorial(n)`
- `C(n, k) = n! / (k! (n-k)!)`
- `P(n, k) = n! / (n-k)!`

For modular combinatorics, compute factorials mod m, then use modular inverse for division.

### 12.6 Counting trailing zeros in n! (LC 172)

Trailing zeros come from factors of 10 = 2·5. There are always more 2s than 5s in n!, so count factors of 5.

`zeros(n!) = floor(n/5) + floor(n/25) + floor(n/125) + ...`

**Why?** Every 5th number contributes at least one factor of 5; every 25th contributes another; etc.

**Time O(log_5 n).**


In [ ]:
def trailing_zeros_factorial(n):
    count = 0
    while n:
        n //= 5
        count += n
    return count

assert trailing_zeros_factorial(0) == 0
assert trailing_zeros_factorial(3) == 0
assert trailing_zeros_factorial(5) == 1
assert trailing_zeros_factorial(25) == 6        # 5 + 1
assert trailing_zeros_factorial(100) == 24
print("trailing_zeros_factorial: OK")


### 12.7 Happy number (LC 202)

Replace n with the sum of squares of its digits, repeat. If you reach 1, it's "happy"; if you loop, it's not. Detect the loop with Floyd's cycle detection — same trick as for linked lists.


In [ ]:
def is_happy(n):
    def square_digits(x):
        total = 0
        while x:
            total += (x % 10) ** 2
            x //= 10
        return total
    slow, fast = n, square_digits(n)
    while fast != 1 and slow != fast:
        slow = square_digits(slow)
        fast = square_digits(square_digits(fast))
    return fast == 1

assert is_happy(19) == True                     # 19 → 82 → 68 → 100 → 1
assert is_happy(2) == False
print("is_happy: OK — Floyd's cycle on digit-square sequence")


### Math practice problems

| Concept | LeetCode |
|---|---|
| GCD of strings | LC 1071 |
| Count primes (sieve) | LC 204 |
| Happy Number | LC 202 |
| Pow(x, n) — fast exponentiation | LC 50 |
| Sqrt(x) | LC 69 |
| Excel column number ↔ letter | LC 168 / 171 |
| Reverse Integer | LC 7 |
| String to Integer (atoi) | LC 8 |
| Palindrome Number (no string conversion) | LC 9 |
| Plus One | LC 66 |
| Add Binary | LC 67 |
| Factorial Trailing Zeros | LC 172 |
| Ugly Number / Ugly Number II | LC 263 / 264 |
| Roman to Integer / Integer to Roman | LC 13 / 12 |
| Fraction to Recurring Decimal | LC 166 |
| Multiply Strings | LC 43 |
| Divide Two Integers (no /) | LC 29 |
| Perfect Squares | LC 279 |


# Part 5 — Bit Manipulation

## 13. Bit operations from scratch

Numbers are stored in binary. Bit manipulation operates on **individual bits** rather than treating the number as one unit. The result: O(1) operations on data structures that would otherwise require O(n).

### 13.1 The six operators

| Operator | Symbol | What it does |
|---|---|---|
| AND | `&` | 1 iff both bits are 1 |
| OR | `|` | 1 iff at least one bit is 1 |
| XOR | `^` | 1 iff bits differ |
| NOT | `~` | flip every bit |
| Left shift | `<<` | multiply by 2 (per shift) |
| Right shift | `>>` | floor-divide by 2 (per shift) |

### Truth tables

```
AND      OR       XOR
0 0 = 0  0 0 = 0  0 0 = 0
0 1 = 0  0 1 = 1  0 1 = 1
1 0 = 0  1 0 = 1  1 0 = 1
1 1 = 1  1 1 = 1  1 1 = 0
```

### 13.2 Two's complement and Python's quirky `~`

Most languages store negative integers in **two's complement**: to negate, flip every bit and add 1. So in 8-bit:
- 5 = 00000101
- -5 = 11111011 (flip → 11111010, then +1 → 11111011)

The two's complement of x equals `2^n - x` for n-bit representation.

**Python is weird** because integers are unbounded. `~x` returns `-(x + 1)` — conceptually "all higher bits are 1." So `~10` returns `-11`, not some giant number.

For bit-twiddling **within a fixed width** (e.g., 32-bit), mask the result: `~x & 0xFFFFFFFF`.


In [ ]:
a = 10        # 1010
b = 13        # 1101

print(f"a & b = {a & b}")              # 1000 = 8
print(f"a | b = {a | b}")              # 1111 = 15
print(f"a ^ b = {a ^ b}")              # 0111 = 7
print(f"~a    = {~a}")                  # -(10+1) = -11
print(f"~a & 0xFF = {~a & 0xFF}")       # masked to 8 bits: 11110101 = 245
print(f"a << 1 = {a << 1}")             # multiply by 2 → 20
print(f"a >> 1 = {a >> 1}")             # floor divide by 2 → 5

# bin() shows binary string (with '0b' prefix)
print(f"bin(10) = {bin(10)}")
print(f"bin(-10) = {bin(-10)}")         # Python represents negative with a minus sign


## 14. The 8 essential bit tricks

These show up in interview problems constantly. Memorize them.

### Trick 1: Check the i-th bit
```python
(n >> i) & 1   # 1 if i-th bit set, else 0
# or equivalently:
n & (1 << i)   # nonzero if set
```

### Trick 2: Set the i-th bit
```python
n | (1 << i)
```

### Trick 3: Clear the i-th bit
```python
n & ~(1 << i)
```

### Trick 4: Toggle the i-th bit
```python
n ^ (1 << i)
```

### Trick 5: Clear the **lowest set bit** (Brian Kernighan's algorithm)
```python
n & (n - 1)
```
**Why?** Subtracting 1 from n flips the lowest set bit to 0 and turns all bits below it to 1. ANDing with n cancels them all. Example: `1010 - 1 = 1001; 1010 & 1001 = 1000`.

This trick is the key to counting set bits in O(set-bits) instead of O(bit-width).

### Trick 6: Isolate the lowest set bit
```python
n & -n
```
**Why?** In two's complement, `-n = ~n + 1`. The result has only the lowest set bit of n.

### Trick 7: Check if n is a power of 2
```python
n > 0 and (n & (n - 1)) == 0
```
**Why?** Powers of 2 have exactly one bit set. Trick 5 clears the only bit → 0.

### Trick 8: XOR is the "find the odd one" tool
- `x ^ x = 0`
- `x ^ 0 = x`
- XOR is commutative and associative.
- So XOR-ing all elements of an array cancels out every pair, leaving only odd-occurrence values.


In [ ]:
# Demonstrate each trick

n = 0b10110                                      # = 22

# 1. Check i-th bit
assert (n >> 1) & 1 == 1                         # bit 1 is set (the "2")
assert (n >> 2) & 1 == 1                         # bit 2 is set (the "4")
assert (n >> 3) & 1 == 0                         # bit 3 is NOT set

# 2. Set bit i
assert n | (1 << 0) == 0b10111

# 3. Clear bit i
assert n & ~(1 << 1) == 0b10100

# 4. Toggle bit i
assert n ^ (1 << 1) == 0b10100
assert n ^ (1 << 0) == 0b10111

# 5. Clear lowest set bit
assert n & (n - 1) == 0b10100                    # 22 → 20 (cleared the "2")

# 6. Isolate lowest set bit
assert n & -n == 0b10                            # the "2"

# 7. Power of 2
def is_pow2(x):
    return x > 0 and (x & (x - 1)) == 0
assert is_pow2(1) == True
assert is_pow2(8) == True
assert is_pow2(7) == False
assert is_pow2(0) == False
print("All 7 bit tricks: OK")


## 15. Bit-manipulation problems

### 15.1 Count set bits / Hamming weight (LC 191)

Two approaches:
1. **Bit-by-bit:** check each of the 32 bits. O(bit-width).
2. **Brian Kernighan:** `n & (n - 1)` clears one bit per iteration. **O(set-bits)** — typically much faster.

Python: `bin(n).count('1')` is the cheating way; the trick is what interviewers want.


In [ ]:
def hamming_weight(n):
    count = 0
    while n:
        n &= n - 1                               # clear lowest set bit
        count += 1
    return count

assert hamming_weight(0b1011) == 3
assert hamming_weight(0b10101010) == 4
assert hamming_weight(0) == 0
assert bin(0b1011).count('1') == 3               # cheating Pythonic way
print("hamming_weight: OK — O(set bits)")


### 15.2 Counting Bits for 0..n (LC 338)

For each i in [0, n], count set bits in i.

**Naive:** call hamming_weight for each → O(n · log n).

**DP trick:** `dp[i] = dp[i >> 1] + (i & 1)`. Reason: `i >> 1` is i with the lowest bit dropped, so it has either same or one-less set bits; adding `i & 1` recovers the LSB. **O(n).**

Or even slicker: `dp[i] = dp[i & (i - 1)] + 1`. Reason: `i & (i-1)` has one fewer set bit than i.


In [ ]:
def count_bits_0_to_n(n):
    dp = [0] * (n + 1)
    for i in range(1, n + 1):
        dp[i] = dp[i >> 1] + (i & 1)
    return dp

assert count_bits_0_to_n(5) == [0, 1, 1, 2, 1, 2]
print("count_bits_0_to_n: OK — O(n) DP")


### 15.3 Single Number (LC 136) — XOR for the unique element

Every element appears twice EXCEPT one. Find the unique one.

**Trick:** XOR every element. Pairs cancel (`x ^ x = 0`), leaving only the singleton.

**Time O(n), Space O(1).** Beats the O(n) space hash set approach.


In [ ]:
def single_number(nums):
    result = 0
    for x in nums:
        result ^= x
    return result

assert single_number([2, 2, 1]) == 1
assert single_number([4, 1, 2, 1, 2]) == 4
print("single_number: OK")


### 15.4 Single Number III (LC 260) — Two unique numbers

Every element appears twice EXCEPT two. Find both.

**Trick.** XOR everything → result = a XOR b (the two unique numbers). Find any set bit in (a XOR b) — that bit differs between a and b. Now partition the array by that bit: a goes in one group, b in the other; XOR each group separately.

(This is the algorithm in your reference. It's beautiful.)


In [ ]:
def two_unique(nums):
    x = 0
    for v in nums:
        x ^= v                                   # x = a ^ b
    # Find lowest set bit — a and b differ in this bit
    diff_bit = x & -x
    a = b = 0
    for v in nums:
        if v & diff_bit:
            a ^= v
        else:
            b ^= v
    return [a, b]

result = two_unique([1, 2, 1, 3, 2, 5])
assert sorted(result) == [3, 5]
result = two_unique([3, 4, 3, 4, 5, 4, 4, 6, 7, 7])
assert sorted(result) == [5, 6]
print("two_unique: OK — XOR partition trick")


### 15.5 Subsets via bitmask (LC 78 — alternate)

For an array of n elements, there are 2ⁿ subsets. Index each subset by a number from 0 to 2ⁿ - 1; bit i tells you whether element i is in the subset.

**O(n · 2ⁿ).** Same complexity as the backtracking version but iterative.


In [ ]:
def subsets_bitmask(nums):
    n = len(nums)
    out = []
    for mask in range(1 << n):                   # 2^n masks
        subset = [nums[i] for i in range(n) if mask & (1 << i)]
        out.append(subset)
    return out

result = subsets_bitmask([1, 2, 3])
assert len(result) == 8
expected = [[], [1], [2], [1,2], [3], [1,3], [2,3], [1,2,3]]
assert sorted(map(tuple, result)) == sorted(map(tuple, expected))
print("subsets_bitmask: OK — iterative subset generation")


### 15.6 Missing Number (LC 268)

Array contains n distinct numbers in [0, n], one missing. Find it.

**XOR trick.** XOR every element AND every expected index `0..n`. Pairs cancel, leaving the missing one.

(Also solvable with arithmetic: `n*(n+1)/2 - sum(arr)`. XOR is overflow-safe and slightly fancier.)


In [ ]:
def missing_number(nums):
    result = len(nums)                           # start with n (we never iterate to n in the loop)
    for i, v in enumerate(nums):
        result ^= i ^ v
    return result

assert missing_number([3, 0, 1]) == 2
assert missing_number([0, 1]) == 2
assert missing_number([9,6,4,2,3,5,7,0,1]) == 8
print("missing_number: OK — XOR cancel")


### 15.7 Sum of two integers without `+` (LC 371)

XOR computes "sum without carry"; AND-then-shift gives the carry. Repeat until no carry.

**This is how hardware adders work.**


In [ ]:
def add_no_plus(a, b):
    # Python ints are unbounded; need to mask to 32 bits to simulate fixed-width
    MASK = 0xFFFFFFFF
    INT_MAX = 0x7FFFFFFF
    while b & MASK:
        carry = (a & b) << 1
        a = (a ^ b) & MASK
        b = carry & MASK
    # If a represents a negative 32-bit value, convert back to Python negative
    return a if a <= INT_MAX else ~(a ^ MASK)

assert add_no_plus(1, 2) == 3
assert add_no_plus(-2, 3) == 1
assert add_no_plus(2, -3) == -1
print("add_no_plus: OK")


### Bit manipulation practice problems

| Concept | LeetCode |
|---|---|
| Number of 1 Bits (Hamming weight) | LC 191 |
| Counting Bits | LC 338 |
| Single Number | LC 136 |
| Single Number II (every elem 3 times except one) | LC 137 |
| Single Number III (two unique) | LC 260 |
| Missing Number | LC 268 |
| Reverse Bits | LC 190 |
| Power of Two | LC 231 |
| Power of Four | LC 342 |
| Sum of Two Integers | LC 371 |
| Hamming Distance | LC 461 |
| Total Hamming Distance | LC 477 |
| Maximum XOR of Two Numbers | LC 421 (uses Trie) |
| Bitwise AND of a Range | LC 201 |
| Subsets (bitmask) | LC 78 |
| Repeated DNA Sequences | LC 187 |


# Part 6 — Synthesis

## 16. Cross-topic decision matrix, complexity table, common bugs

### 16.1 Cross-topic signal → reach-for cheat sheet

| Signal | Topic |
|---|---|
| "Find middle / detect cycle / kth from end" | Linked list slow-fast |
| "Reverse / rearrange list" | Linked list reverse template |
| "All subsets / permutations / combinations" | Backtracking template |
| "N-Queens / Sudoku / Word Search" | Backtracking with pruning |
| "Partition the input into k groups" | Backtracking (or DP if optimization) |
| "Maximize / minimize but locally optimal looks right" | Greedy — but verify with counterexample |
| "Intervals / scheduling" | Sort by end + greedy sweep |
| "Minimum jumps / frontier reach" | Greedy frontier expansion |
| "GCD / LCM / coprime" | Euclidean algorithm |
| "List all primes up to n" | Sieve |
| "x^n efficiently" | Fast exponentiation |
| "Detect cycle in a number sequence" | Floyd's tortoise-hare on the function |
| "Unique element among pairs" | XOR |
| "Generate all subsets, n ≤ 20" | Bitmask or backtracking |
| "Power of two check" | `n & (n-1) == 0` |
| "Count set bits" | Brian Kernighan |

### 16.2 Master complexity table

| Algorithm | Time | Space | Notes |
|---|---|---|---|
| **Linked lists** | | | |
| Reverse list | O(n) | O(1) iter, O(n) rec | |
| Find middle | O(n) | O(1) | Slow-fast |
| Detect cycle | O(n) | O(1) | Floyd's |
| Merge two sorted | O(m + n) | O(1) | Dummy node |
| Sort list (merge sort) | O(n log n) | O(log n) stack | |
| LRU cache get/put | O(1) | O(capacity) | Hashmap + DLL |
| **Backtracking** | | | |
| All subsets | O(n · 2ⁿ) | O(n) stack + O(n · 2ⁿ) out | |
| All permutations | O(n · n!) | O(n) stack + O(n · n!) out | |
| Combinations C(n,k) | O(k · C(n,k)) | O(k) | |
| N-Queens | O(n!) worst | O(n) | Pruned by constraint sets |
| Sudoku solver | Exponential | O(1) board | Practical due to pruning |
| Palindrome partitioning | O(n · 2ⁿ) | O(n) | |
| **Greedy** | | | |
| Activity selection | O(n log n) | O(1) | Sort by end |
| Jump Game | O(n) | O(1) | Frontier |
| Gas Station | O(n) | O(1) | Surplus argument |
| Task scheduler (heap) | O(n log u) | O(u) | u = unique tasks |
| Min arrows | O(n log n) | O(1) | Sort by end |
| **Math** | | | |
| GCD | O(log min(a,b)) | O(1) | Euclidean |
| Primality check | O(√n) | O(1) | 6k±1 |
| Prime factorization | O(√n) | O(log n) factors | |
| Sieve of Eratosthenes | O(n log log n) | O(n) | |
| Fast exponentiation | O(log n) | O(1) iter | |
| Modular fast power | O(log n) | O(1) | `pow(x,n,m)` built-in |
| **Bit manipulation** | | | |
| Hamming weight (Kernighan) | O(set bits) | O(1) | |
| Single number (XOR) | O(n) | O(1) | |
| Two uniques (XOR partition) | O(n) | O(1) | |
| Subsets via bitmask | O(n · 2ⁿ) | O(n · 2ⁿ) out | |
| Power-of-2 check | O(1) | O(1) | |

### 16.3 Interview narration — rehearse these

**Q: Why use a dummy node in linked list problems?**
> The head can require special handling — deleting it, inserting before it, etc. A dummy node before the head means every "real" node has a predecessor, so the same code handles head and middle uniformly. You return `dummy.next` at the end. It removes about half the edge-case bugs from linked list code.

**Q: How does Floyd's cycle detection work and why?**
> Two pointers: slow advances one step, fast advances two. If there's a cycle, both eventually enter it; once both are inside, the gap closes by one per iteration, so they must meet within (cycle length) steps. If fast hits null, there's no cycle. For finding the cycle ENTRY, after they meet, reset slow to head and advance both one step — they meet at the entry due to a clean modular-arithmetic argument: the distance from head to entry equals the distance from the meeting point to the entry, going around the cycle.

**Q: What's the difference between recursion and backtracking?**
> Backtracking is recursion plus state restoration plus early pruning. Pure recursion explores its subtree top-down; backtracking does the same but also (a) undoes its state mutation before returning so siblings see the right state, and (b) prunes branches that can't yield valid solutions. The key signature of backtracking code is the "apply choice → recurse → undo choice" pattern around the recursive call.

**Q: When does greedy work?**
> Two requirements: the greedy choice property (a locally optimal choice extends to a global optimum) and optimal substructure (subproblems combine into the optimum). The way I'd verify it in an interview is to construct an exchange argument: assume my greedy differs from some optimal solution; show I can swap my choice in without making things worse. If I can't make that argument, I probably need DP.

**Q: Why is Euclidean GCD O(log(min(a,b)))?**
> Each step replaces (a, b) with (b, a mod b). The key fact: `a mod b < a/2` whenever b ≤ a/2; otherwise b > a/2 and a mod b = a - b < a/2. So in any two consecutive steps, the larger value at least halves. That gives O(log(min(a,b))) steps. Lamé's theorem makes this precise: the worst case is consecutive Fibonacci numbers, where the algorithm takes about log_φ(n) steps.

**Q: Why is the Sieve of Eratosthenes O(n log log n)?**
> For each prime p ≤ n, we mark about n/p numbers. Summing over primes, the total work is `n · Σ(1/p)` for primes p ≤ n. By Mertens' theorem, that sum grows like log log n. So total work is O(n log log n) — essentially linear for any practical n.

**Q: Why does XOR find the unique element?**
> XOR has three properties that make this work: x ^ x = 0, x ^ 0 = x, and it's commutative and associative. So XOR-ing every element in any order makes all paired elements cancel; only the unsuffered singleton remains. It's O(n) time, O(1) space, beating the hash-set approach in space.

**Q: How does `n & (n - 1)` clear the lowest set bit?**
> Look at any number n in binary. The lowest set bit is some 1 with trailing zeros. Subtracting 1 turns that 1 into 0 and turns all the trailing zeros into 1s — flips everything below and including the lowest set bit. ANDing with n cancels them all because those positions now have a 1-and-0 combination. The result is n with its lowest set bit removed. This is the basis of Brian Kernighan's set-bit-counting algorithm: each iteration removes one bit, so the loop runs exactly (number of set bits) times.

### 16.4 Common bugs and traps

**Linked lists:**
1. Forgetting to save `next` before reassigning `curr.next` → losing the rest of the list.
2. Returning `head` instead of `dummy.next` when head might have been modified.
3. Returning the head of a reversed list as `head` (which is now the tail) instead of `prev`.
4. Off-by-one in two-pointer: does `fast` start at head, head.next, or dummy?
5. Failing on empty list or single-node input.
6. Mutating the input list when an immutable answer was expected.

**Backtracking:**
7. Forgetting to undo the state mutation after recursing.
8. Appending the path itself rather than `path[:]` — all entries in `out` end up pointing to the same (mutated) list.
9. Forgetting to sort before dedup-skip in duplicates variants.
10. Wrong "skip duplicates" condition — should be `i > start` (within this level) not `i > 0`.
11. For combination sum (with reuse), passing `i + 1` instead of `i` in the recursive call.

**Greedy:**
12. Not proving (or even sketching) why greedy works → silently wrong on tricky inputs.
13. Sorting by the wrong key — by start time when you should sort by end time, or vice versa.
14. Confusing "intervals overlap" with "intervals touch" (`<` vs `<=`).
15. Using greedy when DP is required (coin change is the canonical trap).

**Math:**
16. Forgetting `n = abs(n)` before digit-by-digit operations.
17. Off-by-one in Sieve outer bound: `while i * i <= n`, not `< n`.
18. Using `pow(x, n)` instead of `pow(x, n, m)` for modular problems → overflow.
19. Integer overflow in fixed-width languages; Python is safe but the habit `(a * b) % m` then mod again matters.
20. Returning `n*(n+1)//2` instead of `n*(n+1)/2` in trailing zeros — irrelevant in Python's int division but wrong in C/Java.
21. Forgetting that floor division on negatives rounds DOWN (not toward zero) in Python.

**Bit manipulation:**
22. Using `^` for power instead of `**`. (Easy to slip in if you came from C++.)
23. Forgetting that Python's `~x = -(x+1)`, not a fixed-width complement.
24. Mixing up `&` and `|` in masks. AND clears, OR sets.
25. Wrong operator precedence — `n & 1 == 0` is parsed as `n & (1 == 0)` because `==` binds tighter than `&`. Always parenthesize: `(n & 1) == 0`.
26. Shifting by negative values or by more than the bit width — undefined in C, well-defined in Python but loses bits.

### 16.5 The 5-topic master pattern map

| # | Pattern | DS / topic | Canonical |
|---|---|---|---|
| 1 | Slow-fast pointer | Linked list | Middle / cycle detection |
| 2 | Dummy node + multi-pointer | Linked list | Merge / partition / remove |
| 3 | Backtracking template | Recursion | Subsets / permutations / N-queens |
| 4 | Sort + greedy sweep | Greedy | Intervals / arrows |
| 5 | Frontier / surplus | Greedy | Jump / gas station |
| 6 | Euclidean / Sieve | Math | GCD / primes |
| 7 | Fast power | Math | x^n in O(log n) |
| 8 | XOR cancellation | Bits | Single number / two unique |
| 9 | Kernighan / lowest bit | Bits | Count bits / power of 2 |
| 10 | Bitmask state | Bits | Subset generation in O(n · 2ⁿ) |

That's the full toolkit for the long-tail topics. If you can recognize the signal in 60 seconds and write the template within 2-3 minutes — including stating WHY this approach beats the brute force — you're solid.
